In [10]:
import numpy as np
import pandas as pd
import gurobipy as gp

# read xlsx
# path = 'problem_2/problem_2.xlsx'
# try:
#     df = pd.read_excel(path, sheet_name='Solution_Excel', header=None)
# except FileNotFoundError:
#     path = path.split('/')[1]
# df = pd.read_excel(path, sheet_name='Solution_Excel', header=None)

In [26]:
patterns = []
pattern_1 = [
    [0, 1],
    [1, 1],
]

for i in range(100): patterns.append(pattern_1)
pattern_2 = [
    [1, 1],
    [1, 0],
]
for i in range(100): patterns.append(pattern_2)

In [27]:
board = np.array([
    ([0]*20),
])
board = np.zeros((21, 20))

In [28]:
model = gp.Model("doodleFit")
x = {}

placements = np.ndarray(shape=(len(patterns), board.shape[0], board.shape[1]), dtype=object)
for i, pattern in enumerate(patterns):
    for row in range(board.shape[0] - len(pattern) + 1):
        for col in range(board.shape[1] - len(pattern[0]) + 1):
            if ~np.any((board[row:row + len(pattern[0]), col:col + len(pattern)] == 1) & (pattern == 1)):
                placements[i][row][col] = model.addVar(vtype=gp.GRB.BINARY, name=f'pattern_{i}_{row}_{col}')
    
for i, pattern in enumerate(patterns):
    model.addConstr(gp.quicksum(placements[i][row][col]
                    for row in range(board.shape[0]) # TODO look at the range of row and col
                    for col in range(board.shape[1])
                    if placements[i][row][col]) <= 1, 
                    f'one_rotation_piece_{i}')

for i in range(board.shape[0]):
    for j in range(board.shape[1]):
        terms = []
        terms.append(board[i][j])
        for p in range(len(patterns)):
            for ii in range(len(patterns[p])):
                for jj in range(len(patterns[p][0])):
                    if patterns[p][ii][jj] == 1:
                        if i - ii >= 0 and j - jj >= 0 and placements[p][i - ii][j - jj] is not None:
                            terms.append(placements[p][i - ii][j - jj])
        model.addConstr(gp.quicksum(terms) <= 1, f'no_overlap_{i}_{j}')


# Set objective: maximize the count of placed cells
model.setObjective(gp.quicksum(placements[p][i][j] 
                               for p in range(len(patterns))
                               for i in range(board.shape[0])
                               for j in range(board.shape[1])
                               if placements[p][i][j] is not None), gp.GRB.MAXIMIZE)

model.update()
model.getVars()

[<gurobi.Var pattern_0_0_0>,
 <gurobi.Var pattern_0_0_1>,
 <gurobi.Var pattern_0_0_2>,
 <gurobi.Var pattern_0_0_3>,
 <gurobi.Var pattern_0_0_4>,
 <gurobi.Var pattern_0_0_5>,
 <gurobi.Var pattern_0_0_6>,
 <gurobi.Var pattern_0_0_7>,
 <gurobi.Var pattern_0_0_8>,
 <gurobi.Var pattern_0_0_9>,
 <gurobi.Var pattern_0_0_10>,
 <gurobi.Var pattern_0_0_11>,
 <gurobi.Var pattern_0_0_12>,
 <gurobi.Var pattern_0_0_13>,
 <gurobi.Var pattern_0_0_14>,
 <gurobi.Var pattern_0_0_15>,
 <gurobi.Var pattern_0_0_16>,
 <gurobi.Var pattern_0_0_17>,
 <gurobi.Var pattern_0_0_18>,
 <gurobi.Var pattern_0_1_0>,
 <gurobi.Var pattern_0_1_1>,
 <gurobi.Var pattern_0_1_2>,
 <gurobi.Var pattern_0_1_3>,
 <gurobi.Var pattern_0_1_4>,
 <gurobi.Var pattern_0_1_5>,
 <gurobi.Var pattern_0_1_6>,
 <gurobi.Var pattern_0_1_7>,
 <gurobi.Var pattern_0_1_8>,
 <gurobi.Var pattern_0_1_9>,
 <gurobi.Var pattern_0_1_10>,
 <gurobi.Var pattern_0_1_11>,
 <gurobi.Var pattern_0_1_12>,
 <gurobi.Var pattern_0_1_13>,
 <gurobi.Var pattern_0_1_14>,


In [29]:
model.update()
model.optimize()

Gurobi Optimizer version 11.0.0 build v11.0.0rc2 (linux64 - "Ubuntu 22.04.3 LTS")

CPU model: 12th Gen Intel(R) Core(TM) i5-12500H, instruction set [SSE2|AVX|AVX2]
Thread count: 8 physical cores, 16 logical processors, using up to 16 threads

Optimize a model with 620 rows, 76000 columns and 304000 nonzeros
Model fingerprint: 0xed4ba49c
Variable types: 0 continuous, 76000 integer (76000 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 1e+00]
Found heuristic solution: objective 100.0000000
Presolve removed 4 rows and 0 columns
Presolve time: 0.50s
Presolved: 616 rows, 76000 columns, 303400 nonzeros
Variable types: 0 continuous, 76000 integer (76000 binary)

Root simplex log...

Iteration    Objective       Primal Inf.    Dual Inf.      Time
    4781    1.4000000e+02   0.000000e+00   0.000000e+00      5s

Root relaxation: objective 1.400000e+02, 4781 iterations, 4.27 seconds (5

In [85]:
# print("\nThe optimal solutions:")
if model.status == gp.GRB.INFEASIBLE:
    print('The model is infeasible; computing IIS')
    model.computeIIS()
    for c in model.getConstrs():
        if c.IISConstr:
            print('%s' % c.constrName)
if model.status == gp.GRB.OPTIMAL:
    for var in model.getVars():
        print(f"{var.VarName}: {var.X}")

# print(f"The optimal number of courses to take is:{model.objVal}")
# list_of_variables_defined_above = [total_sold, total_quality, expected_total_quality, rev_a, rev_b, rev_c, total_rev, prod_a, prod_b, used_a, used_b, unused_a, unused_b, total_cost, total_profit]
# list_of_variables_defined_above = [unused_a, unused_b]
# for x in list_of_variables_defined_above:
#     print(f"{x}: {x.getValue()}")
# for constr in model.getConstrs():
#     print(f"Constraint: {constr.ConstrName}, Dual Value: {constr.Pi}")

pattern_0_0_0_0: -0.0
pattern_0_0_0_1: -0.0
pattern_0_0_0_2: -0.0
pattern_0_0_0_3: -0.0
pattern_0_0_0_4: -0.0
pattern_0_0_0_5: -0.0
pattern_0_0_0_6: -0.0
pattern_0_0_1_0: -0.0
pattern_0_0_1_1: -0.0
pattern_0_0_1_2: -0.0
pattern_0_0_1_3: -0.0
pattern_0_0_1_4: -0.0
pattern_0_0_1_5: -0.0
pattern_0_0_1_6: -0.0
pattern_0_0_2_0: -0.0
pattern_0_0_2_1: -0.0
pattern_0_0_2_2: -0.0
pattern_0_0_2_3: -0.0
pattern_0_0_2_4: -0.0
pattern_0_0_2_5: -0.0
pattern_0_0_2_6: -0.0
pattern_0_0_3_0: -0.0
pattern_0_0_3_1: -0.0
pattern_0_0_3_2: -0.0
pattern_0_0_3_3: -0.0
pattern_0_0_3_4: -0.0
pattern_0_0_3_5: -0.0
pattern_0_0_3_6: -0.0
pattern_0_0_4_0: -0.0
pattern_0_0_4_1: -0.0
pattern_0_0_4_2: -0.0
pattern_0_0_4_3: -0.0
pattern_0_0_4_4: -0.0
pattern_0_0_4_5: -0.0
pattern_0_0_4_6: -0.0
pattern_0_0_5_0: -0.0
pattern_0_0_5_1: -0.0
pattern_0_0_5_2: -0.0
pattern_0_0_5_3: -0.0
pattern_0_0_5_4: -0.0
pattern_0_0_5_5: -0.0
pattern_0_0_5_6: -0.0
pattern_0_0_6_0: -0.0
pattern_0_0_6_1: -0.0
pattern_0_0_6_2: -0.0
pattern_0_

In [102]:
for c in model.getConstrs():
  # print what the constraint is evaluated to
  if c.Slack < 1e-6:
    print('Constraint %s is active at solution point' % (c.ConstrName))

Constraint one_rotation_piece_0 is active at solution point
Constraint one_rotation_piece_1 is active at solution point
Constraint one_rotation_piece_2 is active at solution point
Constraint one_rotation_piece_3 is active at solution point
Constraint one_rotation_piece_4 is active at solution point
Constraint one_rotation_piece_5 is active at solution point
Constraint one_rotation_piece_6 is active at solution point
Constraint one_rotation_piece_7 is active at solution point
Constraint one_rotation_piece_8 is active at solution point
Constraint one_rotation_piece_9 is active at solution point
Constraint one_rotation_piece_10 is active at solution point
Constraint one_rotation_piece_11 is active at solution point
Constraint one_rotation_piece_12 is active at solution point
Constraint one_rotation_piece_13 is active at solution point
Constraint one_rotation_piece_14 is active at solution point
Constraint one_rotation_piece_15 is active at solution point
Constraint one_rotation_piece_16 i

### write to excel

In [11]:
write_df = df.copy()
write_df.iloc[24, 1] = sum(list([int(val.X) for val in x.values()]))
write_df.iloc[1:8, 1] = list([int(val.X) for val in x.values()])
prerequisites_fullfilled = {}
for course, prerequisite_courses in prerequisites.iterrows():
    prerequisites_fullfilled[course] = 1
    for i, prerequisite_course in enumerate(prerequisite_courses):
        if prerequisite_course != 0 and courses.values[i] != course:
            if x[courses.values[i]].X == 0:
                prerequisites_fullfilled[course] = 0
                break
write_df.iloc[1:8, 7] = list(prerequisites_fullfilled.values())
write_df.iloc[19:22, 2] = [r.getValue() for r in requirements_actually_fulfilled]
write_df.iloc[0:8, 5] = write_df.iloc[0:8, 7]
write_df.iloc[0:8, 6:8] = np.nan

from openpyxl.styles import Font
from openpyxl import load_workbook

with pd.ExcelWriter(path, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
    write_df.to_excel(writer, sheet_name='Solution_Gurobi', index=False, header=False)
book = load_workbook(path)
sheet = book['Solution_Gurobi']


from openpyxl.utils import get_column_letter

source_sheet = book['Solution_Excel']
for i, column in enumerate(source_sheet.columns, start=1):
    letter = get_column_letter(i)
    width = source_sheet.column_dimensions[letter].width
    sheet.column_dimensions[letter].width = width


bold_font = Font(bold=True)

sheet['A25'].font = bold_font
sheet['B25'].font = bold_font

book.save(path)